# SASV: ECAPA-only baseline (ASVspoof 2019 LA)

Scores each SASV trial with **SpeechBrain ECAPA** (same encoder as the app):

1. Build speaker model = mean of enrolment embeddings (official `.trn` lists)
2. `score = cosine(enrol_model, test_embedding)`
3. Compute **SASV-EER / SV-EER / SPF-EER** via `SASVC2022_Baseline.metrics.get_all_EERs`

**Expected pattern (ECAPA alone):** SV-EER relatively low, **SPF-EER high** (spoofs still sound like the claimed speaker), so SASV-EER is pulled up by spoofs.

Use the kernel / venv from `app/server` (speechbrain + scikit-learn).

In [4]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "score_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

import torch
from experiment_lib import DEFAULT_LA, DEFAULT_SASV, RUNS_DIR
from score_lib import score_ecapa_trials

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("LA exists:", DEFAULT_LA.exists())
print("SASV exists:", DEFAULT_SASV.exists())

cuda: True
NVIDIA GeForce RTX 4060 Laptop GPU
LA exists: True
SASV exists: True


## Knobs

- `SMOKE = True` → **500 stratified** dev trials (target/nontarget/spoof mix)
- `SMOKE = False` + `MAX_TRIALS = 0` → **all** dev trials (~29k)
- Prefer `DEVICE = "cuda"` if available

Do **not** take the first N protocol lines — they are all `target` and break EER.

In [5]:
SMOKE = False
SPLIT = "dev"          # or "eval" (eval only after tuning on dev)
MAX_TRIALS = 500 if SMOKE else 0
DEVICE = "cuda"        # falls back to cpu if unavailable
FORCE_CPU = False

## Run ECAPA-only scoring

Writes:

- `runs/ecapa_only_<split>/scores_<split>.csv`
- `runs/ecapa_only_<split>/metrics_<split>.json`

In [6]:
summary = score_ecapa_trials(
    la_root=DEFAULT_LA,
    sasv_root=DEFAULT_SASV,
    split=SPLIT,
    max_trials=MAX_TRIALS,
    device=DEVICE,
    force_cpu=FORCE_CPU,
    output_dir=RUNS_DIR / f"ecapa_only_{SPLIT}",
)
{
    "sasv_eer_%": summary["sasv_eer_percent"],
    "sv_eer_%": summary["sv_eer_percent"],
    "spf_eer_%": summary["spf_eer_percent"],
    "n": summary["num_scored"],
}

Trials: {'target': 1484, 'nontarget': 5768, 'spoof': 22296, 'total': 29548}
Device: cuda


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Enrol dev:   0%|          | 0/10 [00:00<?, ?it/s]

Score trials:   0%|          | 0/29548 [00:00<?, ?it/s]

{
  "system": "ecapa_only",
  "split": "dev",
  "max_trials": 0,
  "num_scored": 29548,
  "key_counts": {
    "target": 1484,
    "nontarget": 5768,
    "spoof": 22296,
    "total": 29548
  },
  "device": "cuda",
  "sasv_eer": 0.15229110512138205,
  "sv_eer": 0.012482662967109896,
  "spf_eer": 0.17909041980507456,
  "sasv_eer_percent": 15.229110512138206,
  "sv_eer_percent": 1.2482662967109897,
  "spf_eer_percent": 17.909041980507457,
  "note": "ECAPA-alone: SV-EER usually low, SPF-EER high (spoofs look like the target)."
}


{'sasv_eer_%': 15.229110512138206,
 'sv_eer_%': 1.2482662967109897,
 'spf_eer_%': 17.909041980507457,
 'n': 29548}

## Interpret

- High **SPF-EER** here is expected: without a CM, many spoofs pass ASV.
- Next: `03_ecapa_plus_cm_sasv.ipynb` adds LFCC or WavLM via score-sum fusion and should **lower SPF-EER / SASV-EER** if the CM helps.

Compare later to SASV baseline score-sum (ECAPA+AASIST) ~**1.7% SASV-EER** on eval (published).